# C4 — Boosting

## Objetivo

Avaliar o `HistGradientBoostingClassifier` sobre a **mesma partição persistida** usada pelo C3. Se o gate `G6_finalistas_tuning` estiver aprovado, executar uma busca **limitada** de hiperparâmetros para **ambos os finalistas** (logística via `busca_C`, boosting via `busca_learning_rate` × `busca_max_iter` × `busca_max_depth`), usando apenas treino e validação.

Este notebook:

- carrega o artefato supervisionado canônico via `load_supervised_artifact`;
- recupera treino e validação via `split_partitions` — não cria nem re-sorteia partições;
- retreina a logística de referência na mesma partição para comparação justa;
- executa a busca limitada **somente** se `G6_finalistas_tuning` estiver `aprovado`;
- respeita `tuning.max_configuracoes_por_modelo` e a métrica principal `pr_auc`;
- **não abre o teste**;
- **não congela finalistas** se `features.final` ainda for `null`;
- não faz calibração nem escolha de limiar (isso pertence ao C5).

## 1. Bootstrap — raiz do projeto e imports

In [ ]:
from pathlib import Path
import itertools
import json
import os
import platform
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display


def encontrar_raiz_projeto() -> Path:
    atual = Path.cwd().resolve()
    for pasta in [atual, *atual.parents]:
        if (pasta / "src").is_dir() and (pasta / "configs").is_dir():
            return pasta
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto.")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("ROOT_DIR:", ROOT_DIR)
print("Python:", sys.version.split()[0])
print("Sistema:", platform.system(), platform.release())

In [ ]:
from src.utils import config as C
from src.utils import env
from src.utils.reproducibility import Run
from src.data.supervised_artifact import (
    load_supervised_artifact,
    split_partitions,
)
from src.evaluation.leakage import check_leakage
from src.evaluation.metrics import metricas_completas
from src.models.logistic import build_logistic_pipeline
from src.models.boosting import build_boosting_pipeline

env.carregar_dotenv()
env.garantir_diretorios()

TABLES_DIR = C.TABLES_DIR
PREDICTIONS_DIR = C.PREDICTIONS_DIR
TABLES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(C.SEED)
print(C.resumo())

## 2. Carregar o artefato supervisionado canônico

Mesma fonte que o C3. O split é **persistido** e reutilizado — este notebook não cria nem re-sorteia partições.

`load_supervised_artifact` já valida, ao ler o Parquet: presença de todas as features, do alvo e da coluna de partição, alvo binário 0/1 e presença das três partições. Não é preciso repetir essas checagens aqui.

In [ ]:
COHORT_KIND = os.getenv("SRAG_COHORT_KIND", "provisional").lower()

df, manifest = load_supervised_artifact(
    ROOT_DIR,
    expected_kind=COHORT_KIND,
)

# Nomes canônicos vêm do manifesto — nada hard-coded aqui.
PARTICAO = manifest["partition_column"]
ALVO     = manifest["target_column"]
FEATURES = list(manifest["feature_columns"])

print("Tipo de coorte:", manifest["cohort_kind"])
print("Dataset:", manifest["dataset_path"])
print("SHA-256:", manifest["dataset_sha256"])
print("Linhas:", manifest["rows"])
print("Colunas:", manifest["columns"])
print("Seed:", manifest["seed"])
print("Teste aberto?", manifest["test_opened"])
print("Partições:", manifest["partition_counts"])
print("Coluna de partição:", PARTICAO)
print("Coluna alvo:", ALVO)
print("Quantidade de features:", len(FEATURES))

display(df.head(3))

## 3. Recuperar as partições via `split_partitions`

`split_partitions(df, manifest)` devolve apenas `(treino, validacao)` — o teste fica fora por construção. O bloco `teste` não é materializado neste notebook.

In [ ]:
treino, validacao = split_partitions(df, manifest)

X_train = treino[FEATURES].copy()
y_train = pd.to_numeric(treino[ALVO], errors="coerce").astype("int8")

X_val = validacao[FEATURES].copy()
y_val = pd.to_numeric(validacao[ALVO], errors="coerce").astype("int8")

assert X_train.index.equals(y_train.index)
assert X_val.index.equals(y_val.index)

display(pd.DataFrame(
    {
        "particao": ["treino", "validacao"],
        "n": [len(treino), len(validacao)],
        "positivos": [int(y_train.sum()), int(y_val.sum())],
        "prevalencia": [float(y_train.mean()), float(y_val.mean())],
    }
))

n_teste = int((df[PARTICAO].astype(str) == "teste").sum())
print(f"Teste: {n_teste:,} linhas — mantido fechado.")

## 4. Checagem de leakage

In [ ]:
check_leakage(X_train)
check_leakage(X_val)

proibidas = set(C.cfg("features", "proibidas") or [])
assert not (set(X_train.columns) & proibidas)
assert not (set(X_val.columns) & proibidas)
print("Nenhuma coluna proibida presente em X_train ou X_val.")

## 5. Logística de referência (retreinada)

Retreinamos a logística na **mesma** partição do C3 para permitir uma comparação direta com o boosting. Esta é uma referência, não um finalista.

> Os hiperparâmetros vêm do YAML: `penalty` é traduzido para `l1_ratio` pela própria `build_logistic_pipeline`.

In [ ]:
from typing import Literal, cast

log_cfg = C.cfg("models", "logistic") or {}

solver = cast(
    Literal["lbfgs", "liblinear", "newton-cg", "newton-cholesky", "sag", "saga"],
    log_cfg.get("solver", "lbfgs"),
)

pipe_log_ref = build_logistic_pipeline(
    C=1.0,
    solver=solver,
    max_iter=int(log_cfg.get("max_iter", 1000)),
    class_weight=log_cfg.get("class_weight", "balanced"),
)

t0 = time.perf_counter()
pipe_log_ref.fit(X_train, y_train)
fit_log_s = time.perf_counter() - t0

prob_log_ref = pipe_log_ref.predict_proba(X_val)[:, 1]
metricas_log_ref = metricas_completas(y_val, prob_log_ref, limiar=0.5)
metricas_log_ref = cast(dict[str, float | str], dict(metricas_log_ref))
metricas_log_ref["modelo"] = "logistica_C1"
metricas_log_ref["duracao_fit_s"] = fit_log_s

display(pd.DataFrame([metricas_log_ref]))

## 6. Boosting — configuração de referência

Primeiro rodamos **uma** configuração neutra, apenas para ter o desempenho do boosting na validação. A busca limitada vem na célula seguinte, condicionada ao gate `G6_finalistas_tuning`.

In [ ]:
boost_cfg = C.cfg("models", "boosting") or {}

# Defaults documentados no YAML; usados como ponto de partida.
DEFAULT_LEARNING_RATE = 0.05
DEFAULT_MAX_ITER = 500
DEFAULT_MAX_DEPTH = None
DEFAULT_MAX_LEAF_NODES = int(boost_cfg.get("max_leaf_nodes") or 31)
DEFAULT_MIN_SAMPLES_LEAF = int(boost_cfg.get("min_samples_leaf") or 20)
DEFAULT_L2 = float(boost_cfg.get("l2_regularization") or 0.0)
DEFAULT_EARLY_STOPPING = bool(boost_cfg.get("early_stopping") or False)
DEFAULT_CLASS_WEIGHT = boost_cfg.get("class_weight") or "balanced"

pipe_boost_ref = build_boosting_pipeline(
    learning_rate=DEFAULT_LEARNING_RATE,
    max_iter=DEFAULT_MAX_ITER,
    max_depth=DEFAULT_MAX_DEPTH,
    max_leaf_nodes=DEFAULT_MAX_LEAF_NODES,
    min_samples_leaf=DEFAULT_MIN_SAMPLES_LEAF,
    l2_regularization=DEFAULT_L2,
    early_stopping=DEFAULT_EARLY_STOPPING,
    class_weight=DEFAULT_CLASS_WEIGHT,
)

t0 = time.perf_counter()
pipe_boost_ref.fit(X_train, y_train)
fit_boost_s = time.perf_counter() - t0

prob_boost_ref = pipe_boost_ref.predict_proba(X_val)[:, 1]

if not np.isfinite(prob_boost_ref).all():
    raise RuntimeError("Probabilidades não finitas (boosting base).")
if not ((prob_boost_ref >= 0) & (prob_boost_ref <= 1)).all():
    raise RuntimeError("Probabilidades fora de [0, 1] (boosting base).")

metricas_boost_ref = metricas_completas(y_val, prob_boost_ref, limiar=0.5)
metricas_boost_ref = cast(dict[str, float | str], dict(metricas_boost_ref))
metricas_boost_ref["modelo"] = "boosting_base"
metricas_boost_ref["duracao_fit_s"] = fit_boost_s

display(pd.DataFrame([metricas_boost_ref]))

## 7. Busca limitada de hiperparâmetros (somente se G6 aprovado)

Regras respeitadas:

- só roda se `gates.G6_finalistas_tuning.status == 'aprovado'`;
- usa **apenas** treino e validação;
- métrica principal = `tuning.metrica_principal` (default `pr_auc`);
- no máximo `tuning.max_configuracoes_por_modelo` combinações por modelo;
- nunca toca no teste.

Os dois finalistas são tunados:

- **logística** — varredura sobre `models.logistic.busca_C`;
- **boosting** — varredura sobre `busca_learning_rate` × `busca_max_iter` × `busca_max_depth`.

Se o gate estiver pendente, as células registram isso e seguem sem buscar — a comparação final usará as configurações de referência.

> Em bases grandes, a busca pode ser cara. O parâmetro `MAX_TREINO_BUSCA` permite subamostrar **somente durante a busca** (a comparação final continua usando o treino completo). Mantenha `None` para usar todo o treino.

In [ ]:
from typing import Literal, cast

# Subamostra opcional durante a busca. None = usa o treino completo.
# Exemplo para bases grandes: MAX_TREINO_BUSCA = 300_000
MAX_TREINO_BUSCA = None


def _subamostrar(X, y, max_n, seed):
    if max_n is None or max_n <= 0 or len(X) <= max_n:
        return X, y
    rng = np.random.default_rng(seed)
    idx = rng.choice(len(X), size=max_n, replace=False)
    return X.iloc[idx].copy(), y.iloc[idx].copy()


def _linha_busca(m, extras: dict) -> dict:
    """
    Constrói a linha do DataFrame de busca com TODAS as métricas relevantes.

    Inclui tanto `ap` quanto `pr_auc` (alias) para que o `sort_values`
    aceite `tuning.metrica_principal` (que pode ser qualquer um dos dois)
    sem KeyError.
    """
    linha = {
        "ap":      float(m["ap"]),
        "pr_auc":  float(m["pr_auc"]),
        "auc":     float(m["auc"]),
        "brier":   float(m["brier"]),
        "f1":      float(m["f1"]),
        "precision": float(m["precision"]),
        "recall":  float(m["recall"]),
    }
    linha.update(extras)
    return linha


def busca_limitada_boosting(
    X_treino, y_treino, X_val, y_val, *,
    seed, max_configs, metrica_principal,
):
    combos_lr    = list(C.cfg("models", "boosting", "busca_learning_rate") or [DEFAULT_LEARNING_RATE])
    combos_iter  = list(C.cfg("models", "boosting", "busca_max_iter")      or [DEFAULT_MAX_ITER])
    combos_depth = list(C.cfg("models", "boosting", "busca_max_depth")     or [DEFAULT_MAX_DEPTH])

    combos = list(itertools.product(combos_lr, combos_iter, combos_depth))

    rng = np.random.default_rng(seed)
    ordem = rng.permutation(len(combos))
    combos = [combos[i] for i in ordem][:max_configs]

    resultados = []
    melhor = None

    for lr, mi, md in combos:
        t0 = time.perf_counter()
        pipe = build_boosting_pipeline(
            learning_rate=float(lr),
            max_iter=int(mi),
            max_depth=(None if md is None else int(md)),
            max_leaf_nodes=DEFAULT_MAX_LEAF_NODES,
            min_samples_leaf=DEFAULT_MIN_SAMPLES_LEAF,
            l2_regularization=DEFAULT_L2,
            early_stopping=DEFAULT_EARLY_STOPPING,
            class_weight=DEFAULT_CLASS_WEIGHT,
        )
        pipe.fit(X_treino, y_treino)
        fit_s = time.perf_counter() - t0
        prob = pipe.predict_proba(X_val)[:, 1]
        m = metricas_completas(y_val, prob, limiar=0.5)

        linha = _linha_busca(m, {
            "learning_rate": float(lr),
            "max_iter":      int(mi),
            "max_depth":     (None if md is None else int(md)),
            "duracao_fit_s": float(fit_s),
        })
        resultados.append(linha)

        score = linha[metrica_principal]
        if melhor is None or score > melhor["score"]:
            melhor = {
                "score": score,
                "metrica_principal": metrica_principal,
                "learning_rate": float(lr),
                "max_iter": int(mi),
                "max_depth": (None if md is None else int(md)),
                "pipeline": pipe,
                "prob_val": prob,
            }

    return pd.DataFrame(resultados), melhor


def busca_limitada_logistica(
    X_treino, y_treino, X_val, y_val, *,
    seed, max_configs, metrica_principal,
):
    valores_C = list(C.cfg("models", "logistic", "busca_C") or [1.0])

    rng = np.random.default_rng(seed)
    ordem = rng.permutation(len(valores_C))
    valores_C = [valores_C[i] for i in ordem][:max_configs]

    resultados = []
    melhor = None

    for c_val in valores_C:
        t0 = time.perf_counter()
        solver_name = log_cfg.get("solver", "lbfgs")
        valid_solvers = {
            "lbfgs",
            "liblinear",
            "newton-cg",
            "newton-cholesky",
            "sag",
            "saga",
        }
        if solver_name not in valid_solvers:
            raise ValueError(
            f"Solver inválido para regressão logística: {solver_name!r}. "
            f"Valores válidos: {sorted(valid_solvers)}"
            )

        solver = cast(
            Literal["lbfgs", "liblinear", "newton-cg", "newton-cholesky", "sag", "saga"],
            solver_name,
        )

        pipe = build_logistic_pipeline(
            C=float(c_val),
            solver=solver,
            max_iter=int(log_cfg.get("max_iter", 1000)),
            class_weight=log_cfg.get("class_weight", "balanced"),
        )
        pipe.fit(X_treino, y_treino)
        fit_s = time.perf_counter() - t0
        prob = pipe.predict_proba(X_val)[:, 1]
        m = metricas_completas(y_val, prob, limiar=0.5)

        linha = _linha_busca(m, {
            "C":             float(c_val),
            "duracao_fit_s": float(fit_s),
        })
        resultados.append(linha)

        score = linha[metrica_principal]
        if melhor is None or score > melhor["score"]:
            melhor = {
                "score": score,
                "metrica_principal": metrica_principal,
                "C": float(c_val),
                "pipeline": pipe,
                "prob_val": prob,
            }

    return pd.DataFrame(resultados), melhor

In [ ]:
G6_STATUS = (C.cfg("gates", "G6_finalistas_tuning") or {}).get("status")
G6_APROVADO = (G6_STATUS == "aprovado")

tuning_cfg = C.cfg("tuning") or {}
METRICA_PRINCIPAL = str(tuning_cfg.get("metrica_principal") or "pr_auc")
MAX_CONFIGS = max(1, int(tuning_cfg.get("max_configuracoes_por_modelo") or 30))

print("G6_finalistas_tuning.status:", G6_STATUS)
print("G6 aprovado?", G6_APROVADO)
print("Métrica principal:", METRICA_PRINCIPAL)
print("Máx. configurações:", MAX_CONFIGS)

# Estado final: começa com as configurações de referência.
resultados_busca_log = None
resultados_busca_boost = None
melhor_config_log = None
melhor_config_boost = None

pipe_log_final = pipe_log_ref
prob_log_final = prob_log_ref
metricas_log_final = dict(metricas_log_ref)

pipe_boost_final = pipe_boost_ref
prob_boost_final = prob_boost_ref
metricas_boost_final = dict(metricas_boost_ref)

if G6_APROVADO:
    print("\nG6 aprovado — executando buscas limitadas (logística + boosting).")
    X_busca, y_busca = _subamostrar(X_train, y_train, MAX_TREINO_BUSCA, SEED)
    print(f"Treino usado nas buscas: {len(X_busca):,} linhas (original: {len(X_train):,}).")

    # ----- Logística: varredura sobre busca_C -----
    print("\n[logística] busca_C:")
    resultados_busca_log, melhor_config_log = busca_limitada_logistica(
        X_busca, y_busca, X_val, y_val,
        seed=SEED,
        max_configs=MAX_CONFIGS,
        metrica_principal=METRICA_PRINCIPAL,
    )
    resultados_busca_log = resultados_busca_log.sort_values(
        [METRICA_PRINCIPAL, "brier"], ascending=[False, True]
    ).reset_index(drop=True)
    display(resultados_busca_log)

    # Re-treina a melhor logística no treino completo
    print("Re-treinando a melhor logística no treino completo...")
    if melhor_config_log is None:
        print("Busca logística não retornou nenhuma configuração válida; mantendo a pipeline de referência.")
        pipe_log_final = pipe_log_ref
    else:
        pipe_log_final = build_logistic_pipeline(
            C=melhor_config_log["C"],
            solver=log_cfg.get("solver", "lbfgs"),
            max_iter=int(log_cfg.get("max_iter", 1000)),
            class_weight=log_cfg.get("class_weight", "balanced"),
        )
    t0 = time.perf_counter()
    pipe_log_final.fit(X_train, y_train)
    fit_log_final_s = time.perf_counter() - t0
    prob_log_final = pipe_log_final.predict_proba(X_val)[:, 1]
    metricas_log_final = metricas_completas(y_val, prob_log_final, limiar=0.5)
    metricas_log_final = {
        **metricas_log_final,
        "modelo": "logistica_tuned",
    }
    metricas_log_final["duracao_fit_s"] = fit_log_final_s
    display(pd.DataFrame([metricas_log_final]))

    # ----- Boosting: varredura sobre learning_rate x max_iter x max_depth -----
    print("\n[boosting] busca_learning_rate x busca_max_iter x busca_max_depth:")
    resultados_busca_boost, melhor_config_boost = busca_limitada_boosting(
        X_busca, y_busca, X_val, y_val,
        seed=SEED,
        max_configs=MAX_CONFIGS,
        metrica_principal=METRICA_PRINCIPAL,
    )
    resultados_busca_boost = resultados_busca_boost.sort_values(
        [METRICA_PRINCIPAL, "brier"], ascending=[False, True]
    ).reset_index(drop=True)
    display(resultados_busca_boost)

    # Re-treina o melhor boosting no treino completo
    print("Re-treinando o melhor boosting no treino completo...")
    if melhor_config_boost is None:
        print("Busca boosting não retornou nenhuma configuração válida; mantendo a pipeline de referência.")
        pipe_boost_final = pipe_boost_ref
    else:
        pipe_boost_final = build_boosting_pipeline(
            learning_rate=melhor_config_boost["learning_rate"],
            max_iter=melhor_config_boost["max_iter"],
            max_depth=melhor_config_boost["max_depth"],
            max_leaf_nodes=DEFAULT_MAX_LEAF_NODES,
            min_samples_leaf=DEFAULT_MIN_SAMPLES_LEAF,
            l2_regularization=DEFAULT_L2,
            early_stopping=DEFAULT_EARLY_STOPPING,
            class_weight=DEFAULT_CLASS_WEIGHT,
        )
    t0 = time.perf_counter()
    pipe_boost_final.fit(X_train, y_train)
    fit_boost_final_s = time.perf_counter() - t0
    prob_boost_final = pipe_boost_final.predict_proba(X_val)[:, 1]
    metricas_boost_final = metricas_completas(y_val, prob_boost_final, limiar=0.5)
    metricas_boost_final = {
        **metricas_boost_final,
        "modelo": "boosting_tuned",
    }
    metricas_boost_final["duracao_fit_s"] = fit_boost_final_s
    display(pd.DataFrame([metricas_boost_final]))
else:
    print("\nG6 pendente — buscas limitadas não executadas.")
    print("A comparação usará as configurações de referência.")

## 8. Comparação na validação — logística vs boosting

Comparamos as versões finais de cada família: tuned (se G6 aprovado) ou de referência.

In [ ]:
def _linha(d, nome):
    return {
        "modelo": nome,
        "ap": float(d["ap"]),
        "pr_auc": float(d.get("pr_auc", d["ap"])),
        "auc": float(d["auc"]),
        "brier": float(d["brier"]),
        "f1": float(d["f1"]),
        "precision": float(d["precision"]),
        "recall": float(d["recall"]),
        "accuracy": float(d["accuracy"]),
        "duracao_fit_s": float(d.get("duracao_fit_s", float("nan"))),
        "n_validacao": int(d["n"]),
    }


comparacao = pd.DataFrame([
    _linha(metricas_log_final,   metricas_log_final["modelo"]),
    _linha(metricas_boost_final, metricas_boost_final["modelo"]),
]).sort_values("ap", ascending=False).reset_index(drop=True)

display(comparacao)

ap_log   = float(comparacao.loc[comparacao["modelo"].str.startswith("logistica"), "ap"].iloc[0])
ap_boost = float(comparacao.loc[comparacao["modelo"].str.startswith("boosting"),  "ap"].iloc[0])
delta_ap = ap_boost - ap_log
print(f"Δ AP (boosting − logística): {delta_ap:+.4f}")

## 9. Predições de validação

Salvamos as predições dos dois modelos na validação para auditoria e inspeção.

In [ ]:
pred_log = pd.DataFrame({
    "y_true": y_val.to_numpy(),
    "y_prob": prob_log_final,
    "modelo": metricas_log_final["modelo"],
    "cohort_kind": manifest["cohort_kind"],
})

pred_boost = pd.DataFrame({
    "y_true": y_val.to_numpy(),
    "y_prob": prob_boost_final,
    "modelo": metricas_boost_final["modelo"],
    "cohort_kind": manifest["cohort_kind"],
})

pred_val = pd.concat([pred_log, pred_boost], ignore_index=True)
pred_val["y_pred"] = (pred_val["y_prob"] >= 0.5).astype(int)

display(pred_val.head())
print("Total de predições de validação:", len(pred_val))

assert pred_val["y_prob"].between(0, 1).all()
assert pred_val["modelo"].nunique() == 2

## 10. Persistência dos artefatos

In [ ]:
timestamp = pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%SZ")
prefixo = f"C4_boosting_{manifest['cohort_kind']}_{timestamp}"

caminho_comparacao = TABLES_DIR / f"{prefixo}_comparacao.csv"
caminho_predicoes = PREDICTIONS_DIR / f"{prefixo}_predicoes_validacao.parquet"
caminho_manifesto = TABLES_DIR / f"{prefixo}_manifesto.json"
caminho_busca_log = TABLES_DIR / f"{prefixo}_busca_logistica.csv"
caminho_busca_boost = TABLES_DIR / f"{prefixo}_busca_boosting.csv"

comparacao.to_csv(caminho_comparacao, index=False, encoding="utf-8-sig")
pred_val.to_parquet(caminho_predicoes, index=False)

if resultados_busca_log is not None:
    resultados_busca_log.to_csv(caminho_busca_log, index=False, encoding="utf-8-sig")
    caminho_busca_log_str = str(caminho_busca_log)
else:
    caminho_busca_log_str = None

if resultados_busca_boost is not None:
    resultados_busca_boost.to_csv(caminho_busca_boost, index=False, encoding="utf-8-sig")
    caminho_busca_boost_str = str(caminho_busca_boost)
else:
    caminho_busca_boost_str = None

features_final_yaml = C.cfg("features", "final")

manifesto_c4 = {
    "cohort_kind": manifest["cohort_kind"],
    "dataset_path": manifest["dataset_path"],
    "dataset_sha256": manifest["dataset_sha256"],
    "feature_columns": FEATURES,
    "target_column": ALVO,
    "partition_column": PARTICAO,
    "n_treino": int(len(treino)),
    "n_validacao": int(len(validacao)),
    "n_teste_fechado": n_teste,
    "g6_status": G6_STATUS,
    "buscas_executadas": bool(G6_APROVADO),
    "metrica_principal": METRICA_PRINCIPAL,
    "max_configs": MAX_CONFIGS,
    "melhor_config_logistica": (
        None if melhor_config_log is None else {
            "C": melhor_config_log["C"],
            "score_metrica_principal": float(melhor_config_log["score"]),
        }
    ),
    "melhor_config_boosting": (
        None if melhor_config_boost is None else {
            "learning_rate": melhor_config_boost["learning_rate"],
            "max_iter":      melhor_config_boost["max_iter"],
            "max_depth":     melhor_config_boost["max_depth"],
            "score_metrica_principal": float(melhor_config_boost["score"]),
        }
    ),
    "limiar_usado": 0.5,
    "limiar_e_diagnostico": True,
    "calibracao_aplicada": False,
    "teste_aberto": False,
    "features_final_congelada_no_yaml": bool(features_final_yaml),
    "finalistas_congelados": False,
    "observacoes": [
        "Resultados dependem da coorte em uso.",
        "Buscas limitadas apenas quando G6 aprovado.",
        "Sem calibração e sem escolha de limiar nesta etapa.",
        "Teste nunca consultado.",
        "Finalistas só serão congelados quando features.final for definida.",
    ],
    "artefatos": {
        "comparacao": str(caminho_comparacao),
        "predicoes_validacao": str(caminho_predicoes),
        "busca_logistica": caminho_busca_log_str,
        "busca_boosting":  caminho_busca_boost_str,
    },
}

caminho_manifesto.write_text(
    json.dumps(manifesto_c4, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

# Round-trip mínimo
comp_lido = pd.read_csv(caminho_comparacao, encoding="utf-8-sig")
pred_lido = pd.read_parquet(caminho_predicoes)
assert len(comp_lido) == len(comparacao)
assert len(pred_lido) == len(pred_val)

print("Comparação       :", caminho_comparacao)
print("Predições        :", caminho_predicoes)
print("Manifesto        :", caminho_manifesto)
if caminho_busca_log_str:
    print("Busca logística  :", caminho_busca_log_str)
if caminho_busca_boost_str:
    print("Busca boosting   :", caminho_busca_boost_str)

## 11. Registro da execução

In [ ]:
with Run(
    modelo="C4_boosting",
    parametros={
        "cohort_kind": manifest["cohort_kind"],
        "g6_status": G6_STATUS,
        "buscas_executadas": bool(G6_APROVADO),
        "metrica_principal": METRICA_PRINCIPAL,
        "max_configs": MAX_CONFIGS,
        "C_logistica_final": (None if melhor_config_log is None else melhor_config_log["C"]),
        "learning_rate_final": (None if melhor_config_boost is None else melhor_config_boost["learning_rate"]),
        "max_iter_final":      (None if melhor_config_boost is None else melhor_config_boost["max_iter"]),
        "max_depth_final":     (None if melhor_config_boost is None else melhor_config_boost["max_depth"]),
        "limiar_diagnostico": 0.5,
        "calibracao_aplicada": False,
        "teste_aberto": False,
    },
    dataset={
        "n_treino": int(len(treino)),
        "n_validacao": int(len(validacao)),
        "n_teste_fechado": n_teste,
    },
    features={
        "n_features": len(FEATURES),
        "usadas": FEATURES,
    },
    notas=(
        "C4 — boosting na mesma partição do C3. "
        "Buscas limitadas (logística e boosting) condicionadas ao gate G6. "
        "Teste não consultado. Limiar 0.5 apenas diagnóstico."
    ),
    dataset_path=Path(manifest["dataset_path"]),
    dataset_sha256=manifest["dataset_sha256"],
) as r:
    r.metrica(
        logistica_ap=float(metricas_log_final["ap"]),
        logistica_auc=float(metricas_log_final["auc"]),
        boosting_ap=float(metricas_boost_final["ap"]),
        boosting_auc=float(metricas_boost_final["auc"]),
        boosting_brier=float(metricas_boost_final["brier"]),
        delta_ap=delta_ap,
    )

print("Run registrado em:", r.pasta)

## 12. Conclusão

Este notebook **não** congela finalistas. Ele apenas fornece evidência de qual família (logística ou boosting) parece mais promissora na validação, respeitando os gates e mantendo o teste fechado.

**Próximo passo:** C5 — calibração e escolha de limiar (após `G7_calibracao` e `G8_limiar` aprovados).

In [ ]:
checklist = {
    "artefato_carregado": "df" in globals() and "manifest" in globals() and not df.empty,
    "particoes_recuperadas": "treino" in globals() and "validacao" in globals(),
    "teste_nao_materializado": "teste" not in globals(),
    "logistica_executada": "metricas_log_final" in globals(),
    "boosting_executado": "metricas_boost_final" in globals(),
    "probabilidades_validas": bool(pred_val["y_prob"].between(0, 1).all()),
    "artefatos_reabertos": True,
    "limiar_nao_escolhido": True,
    "calibracao_nao_aplicada": True,
    "busca_log_condicionada_ao_g6": G6_APROVADO == (resultados_busca_log is not None),
    "busca_boost_condicionada_ao_g6": G6_APROVADO == (resultados_busca_boost is not None),
    "features_final_nao_congelada": not bool(C.cfg("features", "final")),
}

checklist_df = pd.DataFrame(
    {"item": list(checklist.keys()), "concluido": list(checklist.values())}
)
display(checklist_df)

if not all(checklist.values()):
    pendencias = [k for k, v in checklist.items() if not v]
    raise RuntimeError(f"C4 terminou com pendências: {pendencias}")

print("C4 concluído. Teste fechado, finalistas não congelados.")
if not G6_APROVADO:
    print("G6 pendente: as buscas limitadas não foram executadas. "
          "Aprove o gate e rode novamente quando quiser o tuning.")
if not bool(C.cfg("features", "final")):
    print("features.final ainda é null: nenhum finalista foi congelado.")